# ELT Pipeline – Gold Layer (Aggregations & Power BI Export)

**Author:** Jarosław Błaziński  
**Project:** E-Commerce Medallion Architecture (Databricks Community Edition)  
**Stack:** PySpark, Delta Tables, Databricks  

---

## Co robi ta warstwa?

Gold = dane gotowe do raportowania i dashboardów.  
Tworzymy zagregowane tabele zoptymalizowane pod konkretne pytania biznesowe.

```
CSV (source)
     ↓
[BRONZE]  – surowe dane
     ↓
[SILVER]  – oczyszczone dane
     ↓
[GOLD]    ← jesteśmy tutaj → Power BI
```

## 1. Konfiguracja

In [ ]:
import logging
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, round as spark_round, desc,
    sum as spark_sum, avg, count,
    countDistinct, dense_rank, current_timestamp
)
from pyspark.sql.window import Window

spark = SparkSession.builder.appName("ELT_Gold").getOrCreate()
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger("GOLD")

SILVER_PATH = "/tmp/medallion/silver/transactions"
GOLD_PATH   = "/tmp/medallion/gold"

logger.info("Konfiguracja Gold zaladowana")

## 2. Wczytanie danych z Silver

In [ ]:
df_silver = spark.read.format("delta").load(SILVER_PATH)

print(f"Wierszy w Silver : {df_silver.count():,}")
df_silver.show(5, truncate=False)

## 3. Gold #1 – Revenue per kraj i kategoria z rankingiem

**Pytanie biznesowe:** Które kategorie generują największy przychód w każdym kraju?  
Używamy `dense_rank()` żeby uszeregować kategorie w ramach każdego kraju.  
To główna tabela dla Power BI.

In [ ]:
agg_country_category = (
    df_silver
    .groupBy("country", "country_name", "category")
    .agg(
        spark_round(spark_sum("revenue"),     2).alias("total_revenue"),
        spark_round(spark_sum("revenue_vat"), 2).alias("total_revenue_vat"),
        spark_round(avg("amount"),            2).alias("avg_order_value"),
        count("*").alias("transaction_count"),
        countDistinct("customer_id").alias("unique_customers"),
    )
)

window_country = Window.partitionBy("country").orderBy(desc("total_revenue"))

gold_revenue = (
    agg_country_category
    .withColumn("rank_in_country", dense_rank().over(window_country))
    .withColumn("_created_at", current_timestamp())
    .orderBy("country", "rank_in_country")
)

gold_revenue.show(20, truncate=False)

## 4. Gold #2 – Trend miesięczny

**Pytanie biznesowe:** Jak zmieniał się przychód miesiąc do miesiąca w 2024 roku?  
Tabela dla wykresu liniowego w Power BI.

In [ ]:
gold_monthly = (
    df_silver
    .groupBy("year", "month", "country", "country_name")
    .agg(
        spark_round(spark_sum("revenue"), 2).alias("monthly_revenue"),
        count("*").alias("transaction_count"),
        countDistinct("customer_id").alias("unique_customers"),
        spark_round(avg("amount"), 2).alias("avg_order_value"),
    )
    .orderBy("year", "month", "country")
    .withColumn("_created_at", current_timestamp())
)

gold_monthly.show(20, truncate=False)

## 5. Gold #3 – Analiza metod płatności

**Pytanie biznesowe:** Jakie metody płatności dominują w każdym kraju?  
Przydatne dla działu finansowego i optymalizacji checkout.

In [ ]:
gold_payments = (
    df_silver
    .groupBy("country", "country_name", "payment_method")
    .agg(
        count("*").alias("transaction_count"),
        spark_round(spark_sum("revenue"), 2).alias("total_revenue"),
        spark_round(avg("amount"), 2).alias("avg_order_value"),
    )
    .orderBy("country", desc("transaction_count"))
    .withColumn("_created_at", current_timestamp())
)

gold_payments.show(20, truncate=False)

## 6. Zapis wszystkich tabel Gold jako Delta

In [ ]:
tables = {
    "revenue_by_country_category": gold_revenue,
    "monthly_trend":               gold_monthly,
    "payment_methods":             gold_payments,
}

for table_name, df in tables.items():
    path = f"{GOLD_PATH}/{table_name}"
    (
        df.write
        .format("delta")
        .mode("overwrite")
        .save(path)
    )
    logger.info("Zapisano: %s", path)
    print(f"Zapisano: {path}")

print("\nWszystkie tabele Gold zapisane!")

## 7. MERGE INTO – inkrementalna aktualizacja

W produkcji nie nadpisujemy całej tabeli przy każdym uruchomieniu.  
MERGE (upsert) aktualizuje tylko zmienione rekordy – szybciej i bezpieczniej.  
Pokazujemy wzorzec na tabeli revenue.

In [ ]:
from delta.tables import DeltaTable

gold_path_revenue = f"{GOLD_PATH}/revenue_by_country_category"
delta_gold = DeltaTable.forPath(spark, gold_path_revenue)

# Symulujemy nowe dane (np. z kolejnego dnia)
new_data = gold_revenue.limit(5)

(
    delta_gold.alias("target")
    .merge(
        new_data.alias("source"),
        "target.country = source.country AND target.category = source.category"
    )
    .whenMatchedUpdateAll()    # jeśli rekord istnieje – zaktualizuj
    .whenNotMatchedInsertAll() # jeśli nowy rekord – wstaw
    .execute()
)

print("MERGE wykonany!")

# Historia tabeli po MERGE
delta_gold.history().select(
    "version", "timestamp", "operation"
).show(truncate=False)

## 8. Eksport do CSV dla Power BI

Databricks Community Edition nie obsługuje Direct Query z Power BI.  
Eksportujemy Gold jako CSV – podłączamy lokalnie do Power BI Desktop.

In [ ]:
EXPORT_PATH = "/tmp/medallion/export"

exports = {
    "gold_revenue":  gold_revenue,
    "gold_monthly":  gold_monthly,
    "gold_payments": gold_payments,
}

for name, df in exports.items():
    (
        df.coalesce(1)          # jeden plik CSV (łatwiejszy import do Power BI)
        .write
        .option("header", True)
        .mode("overwrite")
        .csv(f"{EXPORT_PATH}/{name}")
    )
    print(f"Wyeksportowano: {EXPORT_PATH}/{name}")

print("\nCSV gotowe do pobrania i podlaczenia w Power BI Desktop")

## 9. Końcowy podgląd – Top 3 kategorie per kraj

In [ ]:
print("=== TOP 3 kategorie per kraj ===")
(
    gold_revenue
    .filter(col("rank_in_country") <= 3)
    .select("country", "category", "total_revenue", "transaction_count", "rank_in_country")
    .orderBy("country", "rank_in_country")
    .show(20, truncate=False)
)

## Podsumowanie Gold Layer

| Tabela Gold | Pytanie biznesowe |
|---|---|
| `revenue_by_country_category` | Które kategorie generują największy przychód per kraj? |
| `monthly_trend` | Jak zmienia się przychód miesiąc do miesiąca? |
| `payment_methods` | Jakie metody płatności dominują per kraj? |

| Krok techniczny | Status |
|---|---|
| 3 tabele Gold z agregacjami | ✅ |
| Window function – ranking per kraj | ✅ |
| Zapis jako Delta Tables | ✅ |
| MERGE INTO (upsert pattern) | ✅ |
| Eksport CSV do Power BI | ✅ |

**Następny krok:** Pobierz pliki CSV z `/tmp/medallion/export/` i podłącz do Power BI Desktop.